  Terminal 1: Gazebo / task board

  cd /home/starforge-sf95/Black-Projects/Project-Automaton/aic-rangers-isaac-training
  export DBX_CONTAINER_MANAGER=docker
  distrobox enter -r aic_eval -- /entrypoint.sh \
    spawn_task_board:=true \
    task_board_x:=0.3 task_board_y:=-0.1 task_board_z:=1.2 \
    task_board_roll:=0.0 task_board_pitch:=0.0 task_board_yaw:=0.785 \
    sfp_mount_rail_0_present:=tr• I checked the existing branches and the current Isaac setup. If I have to choose one lane right now, I
ue sfp_mount_rail_0_translation:=-0.08 \
    sc_mount_rail_0_present:=true sc_mount_rail_0_translation:=-0.09 \
    nic_card_mount_0_present:=true nic_card_mount_0_translation:=0.005 \
    sc_port_0_present:=true sc_port_0_translation:=-0.04 \
    spawn_cable:=true \
    cable_type:=sfp_sc_cable \
    attach_cable_to_gripper:=true \
    ground_truth:=true \
    start_aic_engine:=true \
    shutdown_on_aic_engine_exit:=false \
    skip_model_ready:=true \
    skip_ready_simulator:=true



  Terminal 2: recorder

  cd /home/starforge-sf95/Black-Projects/Project-Automaton/aic-rangers-isaac-training
  unset CONDA_SUBDIR
  bash rangers_teleop/bash/gazebo_record.sh --session-name teleop_session_001

  Terminal 3: Cartesian teleop

  cd /home/starforge-sf95/Black-Projects/Project-Automaton/aic-rangers-isaac-training
  unset CONDA_SUBDIR
  bash rangers_teleop/bash/gazebo_cartesian_teleop.sh

  Terminal 4: episode control
  Start trial:

  cd /home/starforge-sf95/Black-Projects/Project-Automaton/aic-rangers-isaac-training
  bash rangers_teleop/bash/gazebo_episode_start.sh

  Stop trial:

  cd /home/starforge-sf95/Black-Projects/Project-Automaton/aic-rangers-isaac-training
  bash rangers_teleop/bash/gazebo_episode_stop.sh


# Rucha's MISC

 Yep, understood. I’ll explain the mechanical/control-side reasoning more explicitly from here.

  What I found in the code is this:

  - The arm is still running Cartesian impedance control with gravity compensation enabled in aic-rangers-
    isaac-training/aic_bringup/config/aic_ros2_controllers.yaml:26.
  - The controller computes a Cartesian wrench from pose error, then clamps that wrench before converting
    it to joint torques in aic-rangers-isaac-training/aic_controller/src/actions/
    cartesian_impedance_action.cpp:48.
  - In Cartesian mode, the nullspace stiffness is currently all zeros in aic-rangers-isaac-training/
    aic_bringup/config/aic_ros2_controllers.yaml:55, so the arm has no real preference for a “good elbow/
    wrist posture” while tracking TCP pose.
  - The Jacobian pseudo-inverse damping is pretty small: alpha: 0.0005 in aic-rangers-isaac-training/
    aic_bringup/config/aic_ros2_controllers.yaml:81.
  - Payload compensation is zeroed: offset_wrench: [0, 0, 0, 0, 0, 0] in aic-rangers-isaac-training/
    aic_bringup/config/aic_ros2_controllers.yaml:65.

  So the most likely explanations, in order, are:

  1. The wrist/elbow is entering a bad Jacobian region as you pitch down.
      - That would make the Cartesian controller lose authority even though gravity comp is still on.
      - Subjectively, that feels exactly like “it goes limp.”
  2. The controller is saturating available Cartesian wrench.
      - That is especially plausible once the cable, gripper, camera stack, or contact forces start
        loading the wrist at a pitched angle.
      - Gravity comp only handles what the model knows about; unmodeled payload/cable effects have to be
        handled by the Cartesian wrench or offset_wrench.
  3. There is some collision/contact effect once pitched down.
      - Cable drag, self-collision, or scene contact can produce the same symptom.

  What this probably is not:

  - not “policy inference”
  - not aic_engine logic, if you reproduced it in practice mode
  - not simply the old tracking-reset timeout, since we already pushed that out

  The pragmatic way to solve it is not more blind teleop math changes. It is two isolation tests.

  1. Remove cable effects.
      - Run practice mode with attach_cable_to_gripper:=false.
      - If the arm suddenly holds much better past 20 deg, the cable/payload/contact is the trigger.
  2. Remove keyboard teleop from the equation.
      - Use the built-in impedance test script in aic-rangers-isaac-training/aic_bringup/scripts/
        test_impedance.py:1 to command a fixed pitched pose.
      - If a fixed pose also sags, then the problem is controller tuning / singularity, not the keyboard
        teleop loop.

  If those tests confirm controller-side weakness, the fix candidates are:

  - add non-zero Cartesian nullspace stiffness
  - increase Jacobian damping alpha
  - raise maximum_wrench further, likely not just rotational but possibly translational too
  - add a non-zero offset_wrench if the wrist payload/cable is the real source
  - only as a UX safeguard, stop accepting more pitch/roll commands when tcp_error is already high

  My current best guess is:
  bad Jacobian posture + zero nullspace stiffness, with cable/payload making it show up earlier.

  So the right next move is:

  - no-cable test
  - fixed-pose impedance test
  - then patch controller tuning based on which one fails

  That gets us to an actual fix instead of another round of teleop-side guesswork.

